# 00. 기타 — fastText 후보 단어 LLM 필터링 + 유사도 임계값 결정

`04_피처엔지니어링.ipynb`에서 만든 `fasttext_seed_candidates.csv`(13,137행 / seed 30개)를 Claude API로 직접 판정해 노이즈를 걸러내고, **유사도 임계값 θ를 데이터로 역산**한다.

## 핵심 설계

1. **휴리스틱 사전 필터 제거** — 영문 토큰·형태소 조각도 모두 LLM이 직접 판정한다. (이전 스크립트의 `heuristic_noise` 단계 없음)
2. **LLM 입력에 유사도(sim) 포함** — `- 단어 (sim=0.xxx)` 형식. LLM은 라벨만 주고, 임계값은 분포 곡선으로 역산한다.
3. **유사도 구간별 keep 비율 곡선** → "θ 이상 자동 채택, 미만은 LLM 판정" 정책을 데이터로 결정.
4. **30개 seed별 시각화** — seed마다 유사도 분포와 권장 θ를 한눈에 비교.

## 산출
- `data/processed/feature_engineering/fasttext_seed_candidates_reviewed.csv`
- `data/processed/feature_engineering/fasttext_threshold_analysis.csv`
- `reports/figures/00_threshold_*.png` 3종


## 1. 환경 설정 / 경로

In [ ]:
import json
import os
import time
from pathlib import Path

import anthropic
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import rcParams

rcParams["font.family"] = "AppleGothic"
rcParams["axes.unicode_minus"] = False

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FE_DIR = PROJECT_ROOT / "data" / "processed" / "feature_engineering"
SEED_CSV = FE_DIR / "fasttext_seed_candidates.csv"
OUTPUT_CSV = FE_DIR / "fasttext_seed_candidates_reviewed.csv"
THRESHOLD_CSV = FE_DIR / "fasttext_threshold_analysis.csv"
CHECKPOINT = FE_DIR / "fasttext_seed_candidates_reviewed.checkpoint.jsonl"

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "claude-haiku-4-5-20251001"
CHUNK_SIZE = 30
MAX_TOKENS = 2500
MAX_RETRY = 4
RETRY_BASE_SLEEP = 2.0

TARGET_PRECISIONS = (0.90, 0.80, 0.70)
SIM_BIN_WIDTH = 0.05

assert SEED_CSV.exists(), f"입력 파일 없음: {SEED_CSV}"
print("입력 :", SEED_CSV.relative_to(PROJECT_ROOT))
print("출력 :", OUTPUT_CSV.relative_to(PROJECT_ROOT))
print("그림 :", FIG_DIR.relative_to(PROJECT_ROOT))


## 2. API 키 로딩 (.env)

In [ ]:
def _load_dotenv() -> None:
    env_file = PROJECT_ROOT / ".env"
    if not env_file.exists():
        return
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, val = line.split("=", 1)
        os.environ.setdefault(key.strip(), val.strip().strip("\"\'"))


_load_dotenv()
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError(".env 또는 환경변수에 ANTHROPIC_API_KEY가 없습니다")
print("ANTHROPIC_API_KEY 로딩 완료")


## 3. 후보 단어 로딩

`04_피처엔지니어링.ipynb`의 산출물(`fasttext_seed_candidates.csv`)을 그대로 사용한다. 30개 seed × 평균 ~440개 후보 = 약 13,137행.


In [ ]:
df = pd.read_csv(SEED_CSV, encoding="utf-8-sig")
print(f"전체 후보 수: {len(df):,}행")
print(f"seed 수    : {df['seed_term'].nunique()}개")
print()
print("차원 분포:")
print(df["dimension"].value_counts().to_string())
print()
print("유사도 분포:")
print(df["similarity"].describe().round(3).to_string())
df.head()


## 4. ESG 판단 기준 + 시스템 프롬프트

이전 스크립트와 비교한 변경점:

- **휴리스틱 단계 없음** → 영문 토큰·형태소 조각도 LLM이 직접 판정. drop 사유(b/c/d)를 프롬프트에 명시.
- **유사도가 입력으로 들어감** → 참고만 하고 의미 기준으로 판단하라고 지시.
- **응답 형식 보강** — 누락 방지를 위해 "모든 후보를 같은 표기로 포함".


In [ ]:
ESG_CRITERIA = """[E - Environmental 환경]
- 온실가스 배출 및 기후변화 대응 / 에너지 절감·재생에너지 전환
- 대기·수질·토양 오염물질 관리·저감 / 폐기물 감축·재활용·자원순환
- 물·수자원 관리 / 생물다양성·생태계 보전 / 친환경 제품·환경경영체계

[S - Social 사회]
- 인권 보호·차별 금지 / 안전하고 건강한 근로환경
- 공정 채용·평가·보상, 다양성·포용 / 임직원 교육·역량개발
- 협력사 공정거래·동반성장 / 고객 안전·정보보호·만족 / 지역사회 공헌

[G - Governance 지배구조]
- 이사회 독립성·전문성·다양성 / 투명한 의사결정·주주권리 보호
- 윤리경영·반부패·준법 / 리스크 관리·내부통제
- 공시·정보공개 투명성 / 감사기구 독립성"""


SYSTEM_PROMPT = f"""당신은 ESG(환경·사회·지배구조) 보고서 전문 용어 사전을 구축하는 심사 전문가입니다.
fastText 단어 임베딩 유사도로 추출한 후보 단어가, 주어진 seed 개념의 '유효한 확장어'로서
ESG 사전에 등재할 가치가 있는지 판단합니다.

ESG 판단 기준:
{ESG_CRITERIA}

판단 규칙
- keep=true  : 아래 두 조건을 모두 만족
  (1) seed가 속한 ESG 차원(E/S/G)의 기준에 부합하고
  (2) seed 개념의 동의어·하위개념·직접 관련 활동/지표/제도에 해당하는 일반 용어
  예) seed '온실가스' → 배출량, 감축, 온실가스인벤토리, 포집, CDP, SBTi
- keep=false : 위 조건을 만족하지 못하는 경우. 특히 다음은 반드시 drop:
  (a) 형태소 조각·어미·비전형 토큰: 명사로 성립하지 않는 동사/형용사 어간이나
      잘린 단어 (예: 피피한, 방대한, 가까이, 꼽힌, 의사료외, 료재)
  (b) 고유명사·기업명·브랜드명: 특정 회사·제품·기관 이름
      (예: 산수에너지, 인류수에너지, 충천에너지, 군장에너지) → 일반 사전어가 아님
  (c) 외국어 오타·무의미 영문 토큰: ESG 표준 용어가 아닌 외국어 철자 변형
      (예: renewble, energi, runergy, energia) → 단, CDP·SBTi·ESG 등
      널리 통용되는 영문 ESG 약어는 keep 가능
  (d) ESG 무관 일반어: ESG 맥락과 무관한 범용 단어 (예: 성장, 과정, 역사, 산출자)
  (e) 차원 불일치: 다른 ESG 차원에는 맞지만 이 seed의 차원/개념과는 무관한 단어

추가 지침
- 입력에는 각 후보의 fastText 유사도(sim)가 함께 제공됩니다. 유사도는 참고만 하고,
  최종 판단은 의미 기준으로 하십시오. 유사도가 높아도 노이즈면 drop, 낮아도
  명백한 ESG 용어면 keep 하십시오.
- reason은 한 줄(40자 이내)로 핵심 근거만.

응답 형식 — 설명·머리말 없이 JSON 배열만 출력:
[{{"word":"단어","keep":true,"reason":"한줄근거"}}, ...]
입력으로 받은 모든 후보 단어를 빠짐없이 같은 표기로 포함하십시오."""

print(f"SYSTEM_PROMPT 길이: {len(SYSTEM_PROMPT)}자")


## 5. LLM 호출 함수 (유사도 포함)

`make_user_message`에서 각 후보를 `- 단어 (sim=0.xxx)` 형식으로 넘긴다 — 이것이 "라벨만 받고 임계값은 데이터로" 정책의 핵심.


In [ ]:
def make_user_message(dim: str, seed: str, rows: list[tuple[str, float]]) -> str:
    dim_label = {
        "E": "Environmental(환경)",
        "S": "Social(사회)",
        "G": "Governance(지배구조)",
    }.get(dim.upper(), dim)
    word_list = "\n".join(f"- {c} (sim={s:.3f})" for c, s in rows)
    return (
        f"ESG 차원: {dim.upper()} ({dim_label})\n"
        f"Seed 단어: {seed}\n\n"
        f"후보 단어 목록 (fastText 유사도 내림차순):\n{word_list}"
    )


def parse_response_text(text: str) -> list[dict]:
    text = text.strip()
    if "```" in text:
        for part in text.split("```"):
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("["):
                text = part
                break
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1 or end < start:
        raise ValueError("응답에서 JSON 배열을 찾지 못함")
    return json.loads(text[start:end + 1])


def call_llm(client: anthropic.Anthropic, dim: str, seed: str,
             rows: list[tuple[str, float]]) -> list[dict]:
    last_err: Exception | None = None
    for attempt in range(1, MAX_RETRY + 1):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user",
                           "content": make_user_message(dim, seed, rows)}],
            )
            if resp.stop_reason == "max_tokens":
                raise ValueError("출력이 max_tokens에서 잘림")
            text = next((b.text for b in resp.content if b.type == "text"), "")
            return parse_response_text(text)
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRY:
                time.sleep(RETRY_BASE_SLEEP * (2 ** (attempt - 1)))
    raise RuntimeError(f"재시도 {MAX_RETRY}회 실패: {last_err}")


## 6. 체크포인트 (이어하기)

LLM 호출 도중 끊겨도 같은 cell을 다시 실행하면 이미 판정된 후보는 건너뛴다. `.checkpoint.jsonl`에 한 줄씩 저장.


In [ ]:
def load_checkpoint() -> dict[tuple, dict]:
    done: dict[tuple, dict] = {}
    if not CHECKPOINT.exists():
        return done
    for line in CHECKPOINT.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
            done[(rec["dimension"], rec["seed_term"], rec["candidate"])] = rec
        except json.JSONDecodeError:
            continue
    return done


def append_checkpoint(records: list[dict]) -> None:
    with CHECKPOINT.open("a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


done0 = load_checkpoint()
print(f"기존 체크포인트: {len(done0):,}개 판정 완료")


## 7. LLM 필터 실행

휴리스틱 사전 필터 **없이** 모든 후보를 LLM에 보낸다. seed 30개 × 평균 ~440 후보 ÷ 청크 30 ≈ 약 **440 청크**.

테스트할 때는 `LIMIT_SEEDS = 2` 등으로 상위 N개 seed만 처리. 결과가 만족스러우면 `LIMIT_SEEDS = 0`(전체)으로 다시 실행하면 이전에 판정한 분은 건너뛴다.


In [ ]:
LIMIT_SEEDS = 2  # 0 = 전체, >0 = 상위 N seed만 (테스트용)


def run_filter(df: pd.DataFrame, client: anthropic.Anthropic) -> pd.DataFrame:
    df = df.copy()
    done = load_checkpoint()

    target = df
    if LIMIT_SEEDS > 0:
        keep_seeds = (target[["dimension", "seed_term"]]
                      .drop_duplicates().head(LIMIT_SEEDS))
        target = target.merge(keep_seeds, on=["dimension", "seed_term"])
        print(f"LIMIT_SEEDS={LIMIT_SEEDS}: seed {len(keep_seeds)}개만 처리")

    groups = list(target.groupby(["dimension", "seed_term"], sort=False))
    total_chunks = sum((len(g) + CHUNK_SIZE - 1) // CHUNK_SIZE for _, g in groups)
    chunk_idx = fail_chunks = 0

    for (dim, seed), group in groups:
        group = group.sort_values("similarity", ascending=False)
        cand_sim = list(zip(group["candidate"], group["similarity"]))
        for i in range(0, len(cand_sim), CHUNK_SIZE):
            chunk = cand_sim[i:i + CHUNK_SIZE]
            chunk_idx += 1
            if all((dim, seed, c) in done for c, _ in chunk):
                continue
            print(f"[{chunk_idx}/{total_chunks}] {dim}/{seed} "
                  f"({len(chunk)}개)...", end=" ", flush=True)
            try:
                parsed = call_llm(client, dim, seed, chunk)
            except Exception as e:
                print(f"실패: {e}")
                fail_chunks += 1
                continue
            word_map = {str(r.get("word", "")).strip(): r for r in parsed}
            records, matched = [], 0
            for cand, _ in chunk:
                r = word_map.get(cand) or word_map.get(str(cand).strip())
                if r is None:
                    continue
                matched += 1
                rec = {
                    "dimension": dim, "seed_term": seed, "candidate": cand,
                    "keep": bool(r.get("keep")),
                    "reason": str(r.get("reason", ""))[:80],
                }
                done[(dim, seed, cand)] = rec
                records.append(rec)
            append_checkpoint(records)
            keep_n = sum(1 for r in records if r["keep"])
            print(f"keep={keep_n}/{matched}")

    if fail_chunks:
        print(f"\n경고: {fail_chunks}개 청크 실패 → 같은 cell 재실행 시 이어서 처리됨")

    keep_col, reason_col = [], []
    for row in df.itertuples(index=False):
        rec = done.get((row.dimension, row.seed_term, row.candidate))
        if rec is None:
            keep_col.append(pd.NA)
            reason_col.append("")
        else:
            keep_col.append(rec["keep"])
            reason_col.append(rec["reason"])

    df["llm_keep"] = keep_col
    df["llm_reason"] = reason_col
    return df


client = anthropic.Anthropic(api_key=api_key)
result_df = run_filter(df, client)
result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

keep_n = int((result_df["llm_keep"] == True).sum())
drop_n = int((result_df["llm_keep"] == False).sum())
miss_n = int(result_df["llm_keep"].isna().sum())
print(f"\n저장: {OUTPUT_CSV.relative_to(PROJECT_ROOT)}")
print(f"  keep=True : {keep_n:,}개")
print(f"  keep=False: {drop_n:,}개")
print(f"  미판정    : {miss_n:,}개 (재실행하면 이어서 처리됨)")


## 8. 유사도 임계값 분석 — 데이터로 역산

핵심 아이디어: **"유사도 X 이상 후보 중 LLM이 keep한 비율(= 누적 정밀도)"** 곡선을 만들고, 목표 정밀도를 만족하는 가장 낮은 유사도 = 권장 임계값 θ.

LLM은 분포 전체를 보지 못하므로 직접 임계값을 고르게 하지 않고 라벨만 받는다. θ 결정은 통계로.


In [ ]:
def analyze_threshold(df: pd.DataFrame) -> pd.DataFrame:
    judged = df[df["llm_keep"].notna()].copy()
    judged["llm_keep"] = judged["llm_keep"].astype(bool)

    rows = []
    print("=" * 72)
    print("차원별 유사도 임계값 분석 (LLM keep 비율 기반)")
    print("=" * 72)

    for dim in ["E", "S", "G", "ALL"]:
        sub = judged if dim == "ALL" else judged[judged["dimension"] == dim]
        if len(sub) == 0:
            continue
        sub = sub.sort_values("similarity", ascending=False).reset_index(drop=True)
        n = len(sub)
        cum_keep = sub["llm_keep"].cumsum()
        cum_prec = cum_keep / pd.Series(range(1, n + 1))

        print(f"\n[{dim}] 후보 {n:,}개 / 전체 keep율 {sub['llm_keep'].mean():.1%}")
        sub["sim_bin"] = (sub["similarity"] // SIM_BIN_WIDTH) * SIM_BIN_WIDTH
        binned = sub.groupby("sim_bin")["llm_keep"].agg(["mean", "size"])
        print("  유사도구간   keep율    후보수")
        for b, r in binned.sort_index(ascending=False).iterrows():
            bar = "#" * int(r["mean"] * 20)
            print(f"   {b:.2f}+      {r['mean']:5.1%}   {int(r['size']):5d}  {bar}")
            rows.append({"dimension": dim, "type": "bin", "value": round(b, 2),
                         "keep_rate": round(r["mean"], 4),
                         "n_candidates": int(r["size"])})

        for target in TARGET_PRECISIONS:
            ok = cum_prec >= target
            if ok.any():
                last_ok = ok[ok].index.max()
                theta = sub.loc[last_ok, "similarity"]
                kept = int(cum_keep.iloc[last_ok])
                print(f"  목표 keep율 {target:.0%} → 권장 θ≈{theta:.3f} "
                      f"(채택 {kept:,}개 / 정밀도 {cum_prec.iloc[last_ok]:.1%})")
                rows.append({"dimension": dim, "type": f"theta@{target:.0%}",
                             "value": round(float(theta), 4),
                             "keep_rate": round(float(cum_prec.iloc[last_ok]), 4),
                             "n_candidates": kept})
            else:
                print(f"  목표 keep율 {target:.0%} → 만족하는 θ 없음")
    return pd.DataFrame(rows)


analysis_df = analyze_threshold(result_df)
analysis_df.to_csv(THRESHOLD_CSV, index=False, encoding="utf-8-sig")
print(f"\n저장: {THRESHOLD_CSV.relative_to(PROJECT_ROOT)}")
analysis_df.head(20)


## 9. 시각화 ① — 전체/차원별 정밀도 곡선

차원별로 "유사도 X 이상 후보의 keep 비율"이 어떻게 변하는지 본다.

- **단조 감소** → 유사도가 의미 있는 신호. θ를 그어 자동 채택 가능.
- **평평** → 유사도만으로 keep 판별 불가 → LLM 판정을 그대로 사용.


In [ ]:
DIM_COLORS = {"E": "#2ca02c", "S": "#1f77b4", "G": "#ff7f0e", "ALL": "#555555"}

judged = result_df[result_df["llm_keep"].notna()].copy()
judged["llm_keep"] = judged["llm_keep"].astype(bool)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (좌) 누적 정밀도 곡선
ax = axes[0]
for dim in ["E", "S", "G", "ALL"]:
    sub = judged if dim == "ALL" else judged[judged["dimension"] == dim]
    if len(sub) == 0:
        continue
    sub = sub.sort_values("similarity", ascending=False).reset_index(drop=True)
    n = len(sub)
    cum_prec = sub["llm_keep"].cumsum() / pd.Series(range(1, n + 1))
    ax.plot(sub["similarity"], cum_prec, label=f"{dim} (n={n:,})",
            color=DIM_COLORS[dim],
            lw=2.2 if dim == "ALL" else 1.5,
            ls="--" if dim == "ALL" else "-", alpha=0.9)

for target in TARGET_PRECISIONS:
    ax.axhline(target, color="gray", ls=":", alpha=0.5)
    ax.text(judged["similarity"].max(), target, f" {target:.0%}",
            va="center", fontsize=9, color="gray")
ax.invert_xaxis()
ax.set_xlabel("유사도 임계값 θ (이 값 이상 후보만 채택했을 때)")
ax.set_ylabel("누적 keep 비율 (정밀도)")
ax.set_title("① 누적 정밀도 곡선 — θ↑ 일수록 정밀도↑, 채택수↓")
ax.legend(loc="lower left")
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

# (우) 유사도 구간별 keep 비율
ax = axes[1]
bw = SIM_BIN_WIDTH
judged["sim_bin"] = (judged["similarity"] // bw) * bw
for dim in ["E", "S", "G"]:
    sub = judged[judged["dimension"] == dim]
    if len(sub) == 0:
        continue
    binned = sub.groupby("sim_bin")["llm_keep"].mean()
    ax.plot(binned.index + bw / 2, binned.values, marker="o",
            color=DIM_COLORS[dim], label=dim, alpha=0.9, lw=1.6)
ax.set_xlabel("유사도 구간 (폭 0.05)")
ax.set_ylabel("해당 구간의 keep 비율")
ax.set_title("② 유사도 구간별 keep 비율 — 단조 감소면 θ 결정 유효")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
out = FIG_DIR / "00_threshold_precision_curves.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
print(f"저장: {out.relative_to(PROJECT_ROOT)}")
plt.show()


## 10. 시각화 ② — 30개 seed별 유사도 분포 (keep/drop 분리)

seed마다 LLM이 keep/drop한 후보의 유사도 분포가 얼마나 분리되는지 본다. 잘 분리되어 있으면 seed별 개별 θ를, 겹쳐 있으면 LLM 판정을 그대로 쓰는 게 안전.


In [ ]:
seeds_order = list(
    df.drop_duplicates(["dimension", "seed_term"])
      .sort_values(["dimension", "seed_term"])
      [["dimension", "seed_term"]]
      .itertuples(index=False, name=None)
)

n_seeds = len(seeds_order)
ncols = 5
nrows = (n_seeds + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.0, nrows * 2.3),
                         sharex=True, sharey=True)
axes = np.array(axes).reshape(nrows, ncols)

bins = np.linspace(0.40, 1.00, 25)
for idx, (dim, seed) in enumerate(seeds_order):
    ax = axes[idx // ncols, idx % ncols]
    sub = judged[(judged["dimension"] == dim) & (judged["seed_term"] == seed)]
    if len(sub) == 0:
        ax.set_title(f"[{dim}] {seed}\n(미판정)", fontsize=9, color="gray")
        ax.set_facecolor("#f7f7f7")
        continue
    keep_sims = sub.loc[sub["llm_keep"], "similarity"]
    drop_sims = sub.loc[~sub["llm_keep"], "similarity"]
    ax.hist(drop_sims, bins=bins, color="#d62728", alpha=0.55, label="drop")
    ax.hist(keep_sims, bins=bins, color="#2ca02c", alpha=0.65, label="keep")
    keep_rate = sub["llm_keep"].mean()
    ax.set_title(f"[{dim}] {seed}\nn={len(sub)} keep={keep_rate:.0%}",
                 fontsize=9)
    ax.grid(alpha=0.25)

for k in range(n_seeds, nrows * ncols):
    axes[k // ncols, k % ncols].axis("off")

axes[0, 0].legend(loc="upper left", fontsize=8)
fig.suptitle("③ seed별 유사도 분포 (keep=초록 / drop=빨강)", y=1.00, fontsize=12)
plt.tight_layout()
out = FIG_DIR / "00_per_seed_similarity_distribution.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
print(f"저장: {out.relative_to(PROJECT_ROOT)}")
plt.show()


## 11. 시각화 ③ — seed별 권장 임계값 비교

seed마다 "목표 keep율 80%를 만족하는 가장 낮은 유사도(θ@80%)"를 계산해 막대그래프로 본다. seed별로 θ가 크게 다르면 단일 θ보다 seed별 θ가 적합하다는 신호.


In [ ]:
def seed_theta(sub: pd.DataFrame, target: float) -> tuple[float | None, int, int]:
    if len(sub) == 0:
        return None, 0, 0
    sub = sub.sort_values("similarity", ascending=False).reset_index(drop=True)
    n = len(sub)
    cum_keep = sub["llm_keep"].cumsum()
    cum_prec = cum_keep / pd.Series(range(1, n + 1))
    ok = cum_prec >= target
    if not ok.any():
        return None, 0, n
    last_ok = ok[ok].index.max()
    return float(sub.loc[last_ok, "similarity"]), int(cum_keep.iloc[last_ok]), n


rows = []
for dim, seed in seeds_order:
    sub = judged[(judged["dimension"] == dim) & (judged["seed_term"] == seed)]
    theta80, kept80, n = seed_theta(sub, 0.80)
    theta70, kept70, _ = seed_theta(sub, 0.70)
    rows.append({
        "dimension": dim, "seed_term": seed, "n_candidates": n,
        "overall_keep_rate": float(sub["llm_keep"].mean()) if n else 0.0,
        "theta@80%": theta80, "kept@80%": kept80,
        "theta@70%": theta70, "kept@70%": kept70,
    })
seed_summary = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(13, 6))
ss = seed_summary.sort_values(["dimension", "theta@80%"], na_position="last")
labels = [f"[{r.dimension}] {r.seed_term}" for r in ss.itertuples()]
vals = ss["theta@80%"].fillna(0).values
colors = [DIM_COLORS[d] for d in ss["dimension"]]
ax.bar(labels, vals, color=colors, alpha=0.85)
ymax = max(0.85, (np.nanmax(vals) if vals.size else 0) + 0.05)
for i, (v, k) in enumerate(zip(vals, ss["kept@80%"].values)):
    if v > 0:
        ax.text(i, v + 0.005, f"{v:.2f}\nn={int(k)}", ha="center", fontsize=7)
    else:
        ax.text(i, 0.41, "없음", ha="center", fontsize=8, color="red")
ax.set_ylabel("권장 θ (목표 keep율 80% 만족하는 최저 유사도)")
ax.set_title("④ seed별 권장 임계값 — 목표 keep율 80%")
ax.set_ylim(0.4, ymax)
ax.tick_params(axis="x", rotation=70, labelsize=8)
ax.grid(alpha=0.25, axis="y")

handles = [plt.Rectangle((0, 0), 1, 1, color=DIM_COLORS[d], alpha=0.85)
           for d in ["E", "S", "G"]]
ax.legend(handles, ["E", "S", "G"], loc="upper right")

plt.tight_layout()
out = FIG_DIR / "00_per_seed_recommended_theta.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
print(f"저장: {out.relative_to(PROJECT_ROOT)}")
plt.show()

print("\nseed별 임계값 요약:")
print(seed_summary.to_string(index=False))


## 12. 해석 가이드

- **시각화 ①·② 곡선이 단조 감소** → 유사도가 의미 있는 신호. θ를 그어 "θ 이상 자동 채택, 미만은 LLM 판정/검토" 정책이 유효.
- **곡선이 평평** → 유사도만으로 keep 판별 불가 → `llm_keep`을 그대로 최종 채택 기준으로 사용.
- **③ seed별 분포에서 keep/drop이 잘 겹친다** → 단일 θ 부적합. LLM 판정 우선.
- **④ seed별 θ 편차가 크다** → 단일 θ보다 seed별 θ. 단, seed당 후보 수가 적으면 표본 노이즈가 큼 → 차원 평균(`analysis_df`의 `dimension="ALL"` 행)을 안전판으로.

## 산출물

- `data/processed/feature_engineering/fasttext_seed_candidates_reviewed.csv` — 후보별 keep/reason
- `data/processed/feature_engineering/fasttext_threshold_analysis.csv` — 차원·구간별 keep 비율 + 권장 θ
- `reports/figures/00_threshold_precision_curves.png`
- `reports/figures/00_per_seed_similarity_distribution.png`
- `reports/figures/00_per_seed_recommended_theta.png`
